# Lab 11 — Controle com dois graus de liberdade (2DOF)

**Unidade IV — Projeto, sintonia e implementação de PID** · conteúdo 4.3 do PPC

**Objetivos:**
1. Demonstrar o conflito estrutural do controlador 1DOF (servo × regulatório);
2. Implementar PID com **ponderação de referência** (set-point weighting);
3. Implementar a estrutura geral **pré-filtro + realimentação**;
4. Verificar que o 2DOF desacopla as duas respostas.

**Referências:** Åström & Murray (FBS), cap. 11 (seção 2DOF) e cap. 12 · Åström & Hägglund, cap. 5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

def pid_tf(Kp, Ti=np.inf, Td=0.0, N=10):
    """PID ISA com derivada filtrada."""
    C = ct.tf([Kp], [1])
    if np.isfinite(Ti):
        C = C + ct.tf([Kp], [Ti, 0])
    if Td > 0:
        C = C + ct.tf([Kp * Td, 0], [Td / N, 1])
    return C

## 1. O conflito do 1DOF, revisitado

Planta B (FOPDT do Lab 09) com sintonia **agressiva** (boa rejeição de perturbação):

In [ ]:
K_B, tau_B, theta_B = 3.0, 4.0, 1.5
num_p, den_p = ct.pade(theta_B, 5)
G = ct.tf([K_B], [tau_B, 1]) * ct.tf(num_p, den_p)

# sintonia agressiva: lambda pequeno
lam = 0.4 * tau_B
Kp = tau_B / (K_B * (lam + theta_B))
Ti = tau_B
C = pid_tf(Kp, Ti)
print(f"Sintonia agressiva: Kp = {Kp:.3f}, Ti = {Ti}")

# cenário completo: referência em t=0, perturbação em t=25
t = np.linspace(0, 50, 5000)
d = -0.4 * (t >= 25)

T_ry = ct.feedback(C * G, 1)
T_dy = ct.feedback(G, C)
y_1dof = ct.step_response(T_ry, t).outputs + ct.forced_response(T_dy, t, d).outputs

info = ct.step_info(T_ry)
print(f"1DOF agressivo: Mp = {info['Overshoot']:.1f} % ao degrau de referência")

## 2. Solução 1: PID com ponderação de referência (set-point weighting)

$$u = K_p\big(b\,r - y\big) + \frac{K_p}{T_i}\int (r-y)\,dt$$

Somente a parcela **proporcional** vê a referência atenuada por $b$; o integrador continua
vendo o erro completo (senão haveria erro de regime). Implementação por FTs:
a saída é $y = T_{ry}^{(b)}\,r + T_{dy}\,d$ com
$$T_{ry}^{(b)} = \frac{\big(bK_p + \frac{K_p}{T_i s}\big)G}{1 + \big(K_p + \frac{K_p}{T_i s}\big)G}$$

In [ ]:
def pi_2dof_tf(Kp, Ti, b):
    """Retorna (C_r, C_y): ramo da referência e ramo da medição do PI 2DOF."""
    C_r = ct.tf([b * Kp * Ti, Kp], [Ti, 0])   # b Kp + Kp/(Ti s)
    C_y = ct.tf([Kp * Ti, Kp], [Ti, 0])       # Kp + Kp/(Ti s)  (malha)
    return C_r, C_y

plt.figure(figsize=(10, 5))
for b in [1.0, 0.5, 0.0]:
    C_r, C_y = pi_2dof_tf(Kp, Ti, b)
    T_ry_b = ct.feedback(1, C_y * G) * C_r * G   # (C_r G)/(1 + C_y G)
    y_r = ct.step_response(T_ry_b, t).outputs
    y_d = ct.forced_response(T_dy, t, d).outputs   # ramo de perturbação NÃO muda
    plt.plot(t, y_r + y_d, lw=2, label=f'b = {b}')
plt.axhline(1, color='gray', ls='--')
plt.axvline(25, color='gray', ls=':')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Set-point weighting: servo suaviza, regulatório permanece intacto')
plt.legend(); plt.grid(True)
plt.show()

**Resultado central do laboratório:** variar $b$ muda apenas a resposta à referência
(o sobressinal do servo some com $b$ pequeno), enquanto a rejeição de perturbação em
$t = 25$ s é **idêntica** nas três curvas — os polos de malha fechada não dependem de $b$.

## 3. Solução 2: pré-filtro de referência

Estrutura: $u = C(s)\,\big[F(s) r - y\big]$ com $F(s) = \dfrac{1}{T_f s + 1}$.
O pré-filtro suaviza a trajetória vista pela malha.

In [ ]:
plt.figure(figsize=(10, 5))
for Tf in [0.0, 2.0, 5.0]:
    F = ct.tf([1], [Tf, 1]) if Tf > 0 else ct.tf([1], [1])
    T_ry_f = F * T_ry
    y_r = ct.step_response(T_ry_f, t).outputs
    y_d = ct.forced_response(T_dy, t, d).outputs
    plt.plot(t, y_r + y_d, lw=2, label=f'pré-filtro Tf = {Tf} s')
plt.axhline(1, color='gray', ls='--')
plt.axvline(25, color='gray', ls=':')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Pré-filtro: mesmo desacoplamento por outro caminho')
plt.legend(); plt.grid(True)
plt.show()

## 4. Solução 3: feedforward de referência (avançado)

Além de filtrar, podemos **antecipar** o controle: $u_{ff} = \widehat{G}_{ff}(s)\,r$ com
$\widehat{G}_{ff} \approx G^{-1}F_{des}$, deixando à realimentação apenas o erro de modelo.
Como $G^{-1}$ do FOPDT é imprópria, usamos a inversa da parte de 1ª ordem filtrada:

In [ ]:
# resposta desejada: 1ª ordem com constante de tempo T_des (sem inverter o tempo morto)
T_des = 2.0
F_des = ct.tf([1], [T_des, 1])
G_inv_aprox = ct.tf([tau_B, 1], [K_B * T_des, K_B])   # (tau s + 1)/(K (T_des s + 1))

# u = u_ff + u_fb ;  u_ff = G_inv_aprox * F? -> aqui: u_ff direto da referência
# y = G (u_ff + C(F_des r - y))  =>  y = [G Ginv + G C F_des] r / (1 + G C)
T_ff = ct.feedback(1, C * G) * (G * G_inv_aprox + G * C * F_des)

plt.figure(figsize=(10, 5))
y_base = ct.step_response(T_ry, t).outputs
y_ff = ct.step_response(T_ff, t).outputs
plt.plot(t, y_base, lw=2, label='1DOF (só realimentação)')
plt.plot(t, y_ff, lw=2, label='2DOF com feedforward')
resp_des = ct.step_response(F_des * ct.tf(*ct.pade(theta_B, 5)), t)
plt.plot(resp_des.time, resp_des.outputs, 'k:', lw=2, label='trajetória desejada')
plt.axhline(1, color='gray', ls='--')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Feedforward: a saída segue a trajetória projetada')
plt.legend(); plt.grid(True)
plt.show()

## 5. Metodologia de projeto 2DOF (Åström & Murray)

1. **Realimentação primeiro:** projete $C_y$ para robustez e rejeição de perturbação
   (margens, $M_s \le 2$) — pode ser agressivo;
2. **Referência depois:** escolha $b$ (ou $F$, ou feedforward) para a resposta de referência
   desejada — sem sobressinal, sem retocar a malha;
3. Valide os dois canais separadamente e o cenário completo.

---
> **🖼️ Figuras de apoio nos livros:**
> - Ogata, **Figura 8.27** — sistema com controle PI-D (derivada na medição). Cap. 8, **p. 543** (p. 554 do PDF).
> - Ogata, **Figura 8.28** — sistema com controle I-PD (proporcional e derivada na medição — caso $b = 0$). Cap. 8, **p. 544** (p. 555 do PDF).
> - Ogata, **Figuras 8.29 e 8.30** — sistemas de controle com dois graus de liberdade. Cap. 8, **p. 545** (p. 556 do PDF).

## Exercícios (relatório do Lab 11)

**E1.** Prove numericamente que os polos de $T_{ry}^{(b)}$ não dependem de $b$: imprima
`ct.poles` para $b \in \{0;\ 0{,}5;\ 1\}$. Onde $b$ aparece então? (Dica: `ct.zeros`.)

**E2.** Para a sintonia agressiva desta aula, encontre o $b$ que zera o sobressinal do servo
mantendo $t_s \le 15$ s. Reporte a tabela ($b$, $M_p$, $t_s$).

**E3.** Combine set-point weighting com o **PID digital do Lab 10** (o parâmetro `b` da classe
`DigitalPID`) e valide na simulação híbrida com saturação. O 2DOF ajuda também a reduzir o
windup? Explique.

**E4.** Projete um 2DOF completo para o motor CC do Lab 01 (modelo de 2ª ordem):
realimentação PID com PM ≥ 50° e pré-filtro para $M_p = 0$. Apresente servo + regulatório.

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui